# Mid-circuit measurement benchmark on IQM (Amazon Braket, from Qiskit)

This notebook benchmarks how well an IQM quantum processor (Garnet, 20 qubits, or Emerald, 54 qubits)
performs the basic cycle of quantum error correction - **measure an ancilla mid-circuit, then correct
the data qubits based on the outcome** - on every qubit that can act as an ancilla. The circuits are
written in Qiskit and sent to Amazon Braket with the **Qiskit Braket provider**.

Section 0 explains the benchmark and the metrics; section 1 the IQM-specific rules; sections 2-8 run
it: **configure → backend → triplets → plan & cost → submit → harvest → results**.

**Safe by default:** nothing is sent to AWS until you set `SUBMIT = True`. Until then the exact IQM
circuits are run on a local simulator instead (noiseless, since Braket publishes fidelities but no
error model for IQM), under the name `<device>_sim`, with every result file flagged `"simulated": true`.
No AWS credentials are needed for that.

**Requirements**
* `pip install -e ".[iqm]"` from the repository root (or keep the `sys.path` line of the next cell);
* for real hardware: AWS credentials with Amazon Braket access (e.g. `aws configure`). The IQM devices
  are in region `eu-north-1`; set `AWS_DEFAULT_REGION=eu-north-1` if the device is not found.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))          # not needed after `pip install -e .`
DATA, FIGURES = ROOT / "data", ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from qecbench import Chain, Direct
from qecbench.analysis import ancilla_scores, depth_summary, load_results
from qecbench.experiment import build_plan, harvest, print_plan, submit
from qecbench.layout import (ancilla_candidates, ancilla_coverage, grid_layout, min_circuits, pack,
                             qubit_load, select_chains, select_direct_chains)
from qecbench.lrqaoa import ideal_r, random_baseline
from qecbench.plotting import (plot_coverage, plot_depth_spread, plot_device_map, plot_groups,
                               plot_partition)

mpl.rcParams.update({"xtick.labelsize": 14, "ytick.labelsize": 14, "font.size": 14, "axes.linewidth": 2,
                     "axes.labelsize": 15, "lines.linewidth": 2, "legend.fontsize": 12})
%matplotlib inline

## 0. Background

### Why this benchmark

Quantum error correction repeats one cycle over and over: entangle an **ancilla** with a few **data
qubits**, **measure the ancilla in the middle of the circuit**, and act on the outcome with
**feed-forward** (a gate that is applied only if the measured bit is 1), then reuse the ancilla. Gate
and readout fidelities published by vendors do not capture that cycle: they leave out what happens to
the data qubits while the ancilla is read and the classical decision is taken, the reset of the
ancilla, and how all of this behaves when many ancillas are handled at once.

This notebook measures that cycle directly, with a task whose ideal answer is known exactly, on every
qubit of the device that can play the ancilla.

### The task: linear-ramp QAOA on an Ising chain

Take $n$ data qubits on a line and the Hamiltonian

$$H = \sum_{i=0}^{n-2} Z_i Z_{i+1}, \qquad E(x) = \sum_i z_i z_{i+1},\quad z_i = 1 - 2x_i \in \{+1,-1\}.$$

Its ground states are the two alternating bitstrings ($E_{\min} = -(n-1)$) and its worst states the
two uniform ones ($E_{\max} = n-1$). The circuit is the **linear-ramp QAOA** (LR-QAOA): start in
$|+\rangle^{\otimes n}$ and apply $p$ layers

$$e^{-i\gamma_k H}\;\text{then}\;\prod_i R_X(-2\beta_k)_i,\qquad
\gamma_k = \frac{k}{p}\Delta,\quad \beta_k = \frac{p-k+1}{p}\Delta,\quad k = 1..p .$$

The ramp mimics a slow anneal, so on a perfect device the result improves with depth $p$; on a real
device every extra layer also adds errors. Watching the quality versus $p$ separates the two.

### The two implementations of a $ZZ$ rotation

<img src="../figures/paper/1d-chain.jpeg" alt="1D chain: MCM and direct implementations" width="620">

*Left: the chain on the chip - data qubits $q_i$ (white) with an ancilla $a_i$ (yellow) between each
neighbouring pair; the vertical bars mark the $Z_iZ_{i+1}$ terms. Right: the two ways to apply one
such term. **MCM** routes it through the ancilla and a mid-circuit measurement; **direct** applies it
between two adjacent data qubits with no ancilla at all. Both realise the same
$\exp(-i\gamma Z Z)$, and comparing them is what this benchmark does.*

**MCM**, term by term: every $Z_iZ_j$ gets its own ancilla $a$, coupled to both data qubits.

1. The two CNOTs copy the parity $x_i \oplus x_j$ onto the ancilla.
2. $R_Z(2\gamma)$ on the ancilla then multiplies each branch by $e^{-i\gamma z_i z_j}$ - the wanted
   rotation $e^{-i\gamma Z_iZ_j}$ - but the ancilla is still entangled with the data.
3. Measuring the ancilla in the $X$ basis ($H$, then measure) disentangles it. Outcome $m=0$ leaves
   exactly $e^{-i\gamma Z_iZ_j}|\psi\rangle$; outcome $m=1$ leaves $Z_iZ_j\,e^{-i\gamma Z_iZ_j}|\psi\rangle$.
4. **Feed-forward** undoes the extra $Z_iZ_j$ when $m=1$, and the ancilla (left in $|m\rangle$) is
   returned to $|0\rangle$ for the next layer.

**Direct**, the reference: with the two data qubits adjacent, the same rotation is just
`CX, RZ(2γ), CX` on that coupler - one two-qubit rotation, no ancilla, no measurement, no
feed-forward.

Both vendors run the same logical circuits written with CZ gates, since
$\mathrm{CNOT}(d\to a) = H_a\,\mathrm{CZ}\,H_a$ and the inner Hadamards cancel: on the ancilla the MCM
gadget becomes `H, CZ(d_i,a), CZ(d_j,a), RX(2γ), measure`.

### Where it runs on the chip

A **chain** is a path of physical qubits `d0 - a0 - d1 - a1 - ... - d(n-1)`: data and ancillas
alternate, one ancilla per $ZZ$ term. The smallest chain, $n=2$, is the **triplet** `(d1, a, d2)`: one
ancilla, one mid-circuit measurement per layer. Triplets isolate the quality of each ancilla qubit;
longer chains add data qubits that are corrected by two neighbouring ancillas.

Many chains that share no qubit run **in the same circuit**, side by side, so a whole device is
covered in a handful of circuits. A qubit cannot be in two chains of one circuit, so the largest
number of chains any qubit belongs to (its **load**) is a lower bound on the circuits needed.

Three circuit **kinds** can be run on the same chains:

### The two implementations

The same Hamiltonian can be run two ways, and the benchmark is the comparison between them:

| what you run | qubits | how each $ZZ$ rotation is done | kind |
|---|---|---|---|
| **`Chain`** | $n$ data + $n-1$ ancillas, alternating | the gadget above: ancilla measured every layer, data corrected by feed-forward | `mcm` |
| **`Direct`** | $n$ data qubits, adjacent | applied straight to the bond, $\mathrm{RZZ}(2\gamma)$ - no ancilla, no measurement, no feed-forward | `direct` |

Both solve the **same $n$-spin Hamiltonian** at the same depth, so they share the noiseless
reference and their $r$ can be compared directly. Per layer and per bond, `mcm` pays one extra
mid-circuit measurement and its feed-forward; everything else is the same.

`Direct.from_chain(chain)` builds that counterpart. It cannot reuse the chain's own data qubits -
those sit two hops apart with the ancilla between them, and no native two-qubit gate reaches across -
so it takes $n$ *adjacent* qubits of the same path, keeping it on the same patch of the chip: a
triplet `(d1, a, d2)` gives the coupled pair `(d1, a)`. Both go into the same jobs, so they see the
same calibration, and the gap between them is what the measure-and-correct cycle costs.

### What is measured

From the sampled bitstrings of a chain's data qubits:

* **Approximation ratio** $\;r = \dfrac{E_{\max} - \langle E\rangle}{E_{\max} - E_{\min}}$ - 1 when only
  ground states are sampled, 0 for the worst states. Its shot-noise error is stored with it.
* **Random sampling** gives $\langle E\rangle = 0$, so $r_{\rm rand} = 1/2$ exactly, with a shot-noise
  spread $\sigma_{\rm rand} = 1/\big(2\sqrt{(n-1)S}\big)$ for $S$ shots.
* **Noiseless reference** $r_{\rm ideal}(p)$, computed exactly (the next cell plots it).
* **Normalised quality** $\;r_{\rm ovl} = \dfrac{r - r_{\rm rand}}{r_{\rm ideal} - r_{\rm rand}}$ - the
  fraction of the ideal improvement over random guessing that survives the device: 1 = noiseless,
  0 = indistinguishable from random. This is what the device maps show, and what is comparable across
  depths, chain lengths and devices. At small $p$ the denominator is small, so $r_{\rm ovl}$ is noisier
  there.

## 1. How IQM executes feed-forward, and what that allows

On Braket, IQM exposes feed-forward as two instructions:

* `measure_ff(key)` - measure a qubit mid-circuit and store the bit under an integer `key`;
* `cc_prx(θ, φ, key)` - apply the rotation $\mathrm{PRX}(\theta,\varphi)$ **only if** the bit stored
  under `key` is 1.

The gadget of section 0 is built from these and the two native gates, $\mathrm{PRX}$ and CZ. A
conditional $Z$ becomes two conditional rotations, $Z \propto \mathrm{PRX}(\pi,\pi/2)\,\mathrm{PRX}(\pi,0)$,
and the ancilla is returned to $|0\rangle$ by a conditional $\mathrm{PRX}(\pi,0)$ on itself.

Four rules apply (Amazon Braket developer guide, *Dynamic circuits on IQM devices*):

1. every `measure_ff` key is unique, and a `cc_prx` comes after the `measure_ff` it reads;
2. **within one circuit, a qubit's feed-forward can be controlled by only one qubit** (itself or one
   other);
3. **control only works inside a feed-forward group** - the chip is split into fixed groups of qubits
   (2 on Garnet, 4 on Emerald), and an ancilla can only correct qubits of its own group;
4. the program is sent **verbatim**: no compiler touches it, which is why it is written in native gates.

Rules 1 and 4 are satisfied by construction. Rule 3 removes every coupler between groups before the
triplets are chosen. **Rule 2 limits chain length:** in a chain `d0 - a0 - d1 - a1 - d2` the middle data
qubit `d1` must be corrected by both `a0` and `a1`, which is forbidden. So on IQM the
measurement-based benchmark runs on **triplets** (`N_DATA = 2`, one ancilla per data pair). The
`Direct` implementation has no feed-forward at all and runs at any length; section 5 shows the
rejection message for the measurement-based version.

**Billing** is per task (one circuit) plus per shot, independent of circuit duration. Braket
publishes gate fidelities for IQM but no durations.

## 2. Configuration - the only cell you normally edit

* `QPU_NAME` - `iqm_garnet` or `iqm_emerald`.
* `DEPTHS`, `SHOTS`, `DELTA` - LR-QAOA depths, shots per circuit and ramp amplitude $\Delta$.
* `DIRECT_REFERENCE` - also run the ancilla-free `Direct` implementation on the same qubits, in the
  same tasks (table in section 0). It roughly doubles the cost and is what makes the result
  interpretable.
* `ALLOWED_QUBITS` / `ANCILLAS` - restrict to part of the chip, or to certain ancillas.
* `BUFFER`, `MAX_PER_BATCH` - packing density (more circuits, less crowding); keep fixed when comparing.
* `SUBMIT` - `True` sends the tasks to AWS and spends Braket credits; `False` simulates them locally.

In [ ]:
QPU_NAME = "iqm_garnet"         # iqm_garnet | iqm_emerald
N_DATA   = 2                    # the measurement-based kind needs triplets on IQM (section 1)
DEPTHS   = [3, 6, 9, 12]
SHOTS    = 500
DELTA    = 0.5
DIRECT_REFERENCE = True         # also run the ancilla-free Direct version on the same qubits

ALLOWED_QUBITS = None           # e.g. FF_GROUPS[QPU_NAME][1] for one feed-forward group
ANCILLAS       = None
BUFFER         = False
MAX_PER_BATCH  = None

SUBMIT = False                  # True sends the tasks to AWS; False simulates them locally

What a perfect device would give - no hardware involved. The grey band is where random guessing
lands with the number of shots configured below; a measurement inside it carries no signal.

In [ ]:
ps = list(range(1, 21))
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for n in (2, 3, 5):
    chain = Chain(range(2 * n - 1))
    ax.plot(ps, [ideal_r(chain, p, DELTA) for p in ps], "o-", ms=4, label=f"noiseless, n_data = {n}")
sigma = random_baseline(Chain((0, 1, 2)), SHOTS)[1]
ax.axhspan(0.5 - 3 * sigma, 0.5 + 3 * sigma, color="gray", alpha=0.3, lw=0,
           label=f"random guessing ±3σ (triplet, {SHOTS} shots)")
ax.set_xlabel("LR-QAOA depth $p$")
ax.set_ylabel("$r$")
ax.set_xticks([1, 5, 10, 15, 20])
ax.legend(frameon=False, fontsize=10)
triplet = Chain((0, 1, 2))
print("noiseless r of a triplet at the configured depths:",
      {p: round(ideal_r(triplet, p, DELTA), 3) for p in DEPTHS})

## 3. Backend, coupling map and feed-forward groups

Loads the chip's coupling map (stored with the package, so planning works without AWS access) and the
feed-forward groups. The printout compares all geometric triplets with those whose three qubits lie in
one group - only the latter can run. The map colours each qubit by its group; couplers between groups
are not drawn. With `SUBMIT = True` a calibration snapshot (readout fidelity `fRO`, single-qubit and CZ
fidelities, T1/T2) is saved to `data/calibration/`; local simulation needs no AWS access and skips it.

In [ ]:
from qecbench.backends import FF_GROUPS, IQMBackend
from qecbench.layout import enumerate_chains, restrict_to_groups

backend = IQMBackend(QPU_NAME, local=not SUBMIT, seed=1)
G = backend.coupling_graph()                   # refresh=True re-reads it from AWS
pos = grid_layout(G)
groups = backend.feedforward_groups()
G_ff = restrict_to_groups(G, groups)            # couplers usable for feed-forward
print(f"{QPU_NAME}: {G.number_of_nodes()} qubits, {G.number_of_edges()} couplers, "
      f"{G_ff.number_of_edges()} inside feed-forward groups")
print(f"{len(enumerate_chains(G, 2))} geometric triplets, {len(enumerate_chains(G_ff, 2))} inside one group")

NODE_SIZE, FONT_SIZE, MAP_FIGSIZE = (700, 10, (9, 8)) if G.number_of_nodes() < 30 else (420, 8, (11, 10))
fig, ax = plt.subplots(figsize=MAP_FIGSIZE)
plot_groups(G_ff, groups, pos=pos, ax=ax, node_size=NODE_SIZE, font_size=FONT_SIZE,
            title=f"{QPU_NAME} - feed-forward groups")
cal = None if backend.simulated else backend.calibration(save_dir=DATA / "calibration")

## 4. Choose the triplets

Every qubit with at least two neighbours **in its own group** becomes the ancilla of **exactly one**
triplet. Where a qubit has several neighbour pairs, the pair is chosen so that no qubit ends up in too
many triplets - that keeps the number of circuits at its lower bound (the largest number of triplets
sharing one qubit). With a calibration loaded, ties go to the best-calibrated couplers.

In [ ]:
cost = (lambda chain: backend.error_budget(chain, cal)) if cal else None
chains = select_chains(G_ff, N_DATA, allowed_qubits=ALLOWED_QUBITS, ancillas=ANCILLAS, cost=cost,
                       buffer=BUFFER)
assert chains, "no chain fits inside a feed-forward group - widen ALLOWED_QUBITS / ANCILLAS"

region = set(G_ff.nodes) if ALLOWED_QUBITS is None else set(ALLOWED_QUBITS)
cands, cover, load = ancilla_candidates(G_ff, ALLOWED_QUBITS, ANCILLAS), ancilla_coverage(chains), qubit_load(chains)
batches = pack(chains, G_ff, buffer=BUFFER)
missed = sorted(set(cands) - set(cover))

print(f"device      : {QPU_NAME}, {G.number_of_nodes()} qubits; {len(cands)} can host a "
      f"mid-circuit measurement inside their feed-forward group")
print(f"chains      : {len(chains)} of {N_DATA} data qubits: {[c.qubits for c in chains]}")
print(f"coverage    : {len(cover)}/{len(cands)} ancilla-capable qubits benchmarked"
      + (f", missed {missed}" if missed else ", once each"))
print(f"qubits used : {len(load)}/{len(region)}   ({len(cover)} as ancilla, "
      f"{len(load) - len(cover)} data only, {len(region) - len(load)} untouched)")
print(f"parallelism : {len(batches)} circuits per depth (lower bound {min_circuits(chains)}), "
      f"{'/'.join(str(len(b)) for b in batches)} chains running at once")

if cal:
    budget = {c: backend.error_budget(c, cal) for c in chains}
    rank = sorted(budget, key=budget.get)
    print(f"calibration : predicted error per layer, median {np.median(list(budget.values())):.4f}"
          f"   best {rank[0].qubits} {budget[rank[0]]:.4f}   worst {rank[-1].qubits} {budget[rank[-1]]:.4f}")
    flagged = backend.flag_instances(chains, cal)
    print(f"              {len(flagged)} chain(s) touch hardware outside the usual thresholds"
          + (f", e.g. {flagged[0][0].qubits}: {flagged[0][1]}" if flagged else ""))

# the same Hamiltonian without the ancilla, on n_data adjacent qubits of each chain's own path
if DIRECT_REFERENCE:
    direct = sorted({Direct.from_chain(c) for c in chains})   # neighbouring chains can share a pair
    instances = chains + direct
    print(f"instances   : {len(instances)} = {len(chains)} mcm + {len(direct)} direct"
          + (f"   ({len(chains) - len(direct)} chain(s) share a direct counterpart)"
             if len(direct) < len(chains) else ""))
else:
    instances = list(chains)
    print(f"instances   : {len(instances)} = {len(chains)} mcm, no direct reference")

## 5. Plan and cost - before anything is sent

`build_plan` packs qubit-disjoint triplets into circuits, checks every circuit against the rules of
section 1, and builds it in native gates for every depth and kind. The table prices it at the Braket
rate for the device (task fee plus shot fee, one task per circuit); credits are consumed 1:1 with USD.
If the rate card changes, edit `PRICING` in `qecbench/backends/iqm.py`.

In the panels below, each panel is one circuit: coloured paths are the triplets that run together,
filled nodes their ancillas, hollow nodes their data qubits.

In [ ]:
plan = build_plan(backend, instances, depths=DEPTHS, shots=SHOTS, delta=DELTA,
                  buffer=BUFFER, max_per_batch=MAX_PER_BATCH)
print_plan(plan, backend)
batches = plan["batches"]["mcm"]
fig, axes = plot_partition(G, batches, pos=pos, ncols=3, node_size=NODE_SIZE // 2,
                           title=f"{QPU_NAME} - the {len(batches)} mcm circuits of one depth")

Rule 2 of section 1 in action: planning one 3-qubit chain with the measurement-based kind is refused,
and the message names the data qubit that would need two controllers.

In [ ]:
try:
    build_plan(backend, select_chains(G_ff, 3, restarts=1)[:1], depths=[1], shots=10)
except ValueError as exc:
    print(str(exc).splitlines()[1])

## 6. Inspect the Braket program, then submit

The first cell prints the program that Braket receives for one triplet at $p = 1$, in OpenQASM 3. Things
to recognise:

* `#pragma braket verbatim` and the `box { ... }` - the verbatim block (rule 4);
* `prx` / `cz` on physical qubits `$<label>` - the native gates;
* `measure_ff(0)` on the ancilla, followed by the `cc_prx(..., 0)` operations that read key 0: one on
  the ancilla (its reset) and two on each data qubit (the conditional $Z$);
* the final `measure` of the two data qubits only - the mid-circuit results themselves are never
  returned by the device.

The second cell runs the plan: with `SUBMIT = True` it sends one Braket task per circuit, with
`SUBMIT = False` it simulates them locally instead. The manifest is saved after every task either way.

In [ ]:
one = plan["instances"][:1]
program = backend.to_braket(backend.build(one, 1, DELTA, "mcm"))
print(program.to_ir("OPENQASM").source)

In [ ]:
manifest_path = submit(plan, backend, manifest_dir=DATA / "manifests")

## 7. Harvest

Submission writes a **manifest** (`data/manifests/<backend>/`): which chains sit in which circuit of
which job. Harvesting needs only that file, so it can happen hours or days later, in a new session. The
cell takes the newest real manifest of this device (paste another path to harvest an older run),
downloads finished jobs, skips unfinished ones, and writes one result file per chain and depth.
Running it again only adds what is new.

Results are filed as `data/results/<backend>/<structure>/<kind>/`, e.g.
`ibm_phoenix/chain/mcm/` and `ibm_phoenix/chain/direct/`, so a long campaign stays navigable and a
surface-code run later lands beside the chains rather than in the same heap.

**One file per run**, not per chain: each holds a `run` header (manifest, shots, depths, whether it was
simulated and under what noise) and a `results` list with every chain and depth. Harvesting the same
manifest again merges into that file, so jobs that finish later are simply added.

Each result file stores the raw counts of the chain's data qubits, $r$ with its error, the random and
noiseless references and $r_{\rm ovl}$, plus the job id and circuit it came from.

In [ ]:
manifests = sorted((p for p in (DATA / "manifests").rglob(f"*_{backend.name}_*.json")
                    if not p.name.endswith("_memory.json")), key=lambda p: p.name)   # memory runs: their own notebook
assert manifests, "no manifest yet - run the submit cell of section 5 first"
manifest_path = manifests[-1]
print("harvesting", manifest_path.name)
saved = harvest(manifest_path, backend, data_dir=DATA / "results")

## 8. Results

**Depth sweep.** Left: $r$ versus $p$ - one faint line per instance, the median (points) and the
inter-quartile band, against the noiseless curve (dashed) and the random-guessing band (grey). Right:
the same data as $r_{\rm ovl}$, which divides out the ramp so that a perfect device would sit at 1 for
every $p$. A steady decline of $r_{\rm ovl}$ with $p$ is the cost of each extra layer.

Blue is the measurement-based chain, orange the direct implementation on the same qubits (if it was
run). **The gap between them is the result**: the part of the decay that belongs to the mid-circuit
measurement and the feed-forward rather than to the gates and idling of those qubits.

In [ ]:
MAP_KIND = "mcm"
results = load_results(DATA / "results", backend.name, kind=MAP_KIND, n_data=N_DATA)
assert results, f"no {MAP_KIND} results for {backend.name} yet - submit and harvest first"
# the direct implementation solves the same Hamiltonian, so it has the same number of data qubits
direct = load_results(DATA / "results", backend.name, kind="direct", n_data=N_DATA)
print(f"{len(results)} chains and {len(direct)} direct instances with data")
print(f"{'p':>4} {'median r':>9} {'IQR':>17} {'median r_ovl':>13} {'n':>5}")
r_sum, o_sum = depth_summary(results, "r"), depth_summary(results, "r_ovl")
for p in r_sum:
    print(f"{p:>4} {r_sum[p][0]:>9.3f}  [{r_sum[p][1]:.3f}, {r_sum[p][2]:.3f}] {o_sum[p][0]:>13.3f} {r_sum[p][3]:>5}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
plot_depth_spread(results, "r", ax=axes[0], label="chains (mcm)")
plot_depth_spread(results, "r_ovl", ax=axes[1], label="chains (mcm)")
if direct:
    plot_depth_spread(direct, "r", ax=axes[0], color="#b8560f", label="direct", reference=False)
    plot_depth_spread(direct, "r_ovl", ax=axes[1], color="#b8560f", label="direct")
    d_sum = depth_summary(direct, "r_ovl")
    print("\nwhat the measure-and-correct cycle costs, in r_ovl:")
    print(f"{'p':>4} {'mcm':>8} {'direct':>8} {'gap':>8}")
    for p in sorted(set(o_sum) & set(d_sum)):
        print(f"{p:>4} {o_sum[p][0]:>8.3f} {d_sum[p][0]:>8.3f} {d_sum[p][0] - o_sum[p][0]:>+8.3f}")
axes[1].set_ylim(-0.2, 1.1)
fig.tight_layout()
fig.savefig(FIGURES / f"{backend.name}_chain{N_DATA}_{MAP_KIND}_depth.pdf", bbox_inches="tight")

**Device map.** Every ancilla qubit coloured by $r_{\rm ovl}$ of its chain at one depth (the best
chain, if a qubit was the ancilla of several). Grey qubits were not benchmarked as ancillas. Dark
spots are the qubits that handle mid-circuit measurement and feed-forward worst - the ones to avoid as
syndrome qubits. The lists below name the best and worst chains.

In [ ]:
PLOT_DEPTH = DEPTHS[1] if len(DEPTHS) > 1 else DEPTHS[0]
scores = ancilla_scores(results, depth=PLOT_DEPTH, value="r_ovl")
fig, ax = plt.subplots(figsize=MAP_FIGSIZE)
plot_device_map(G, scores, pos=pos, ax=ax, vmin=0, vmax=1, node_size=NODE_SIZE, font_size=FONT_SIZE,
                label=r"$r_{\rm ovl} = (r - r_{\rm rand}) / (r_{\rm ideal} - r_{\rm rand})$",
                title=f"{backend.name} - quality per ancilla, n_data = {N_DATA}, p = {PLOT_DEPTH}")
fig.savefig(FIGURES / f"{backend.name}_chain{N_DATA}_{MAP_KIND}_map.pdf", bbox_inches="tight")

ranked = sorted(((by[PLOT_DEPTH]["r_ovl"], inst) for inst, by in results.items() if PLOT_DEPTH in by),
                key=lambda x: -x[0])
print("best :", [(str(i.qubits), round(v, 3)) for v, i in ranked[:5]])
print("worst:", [(str(i.qubits), round(v, 3)) for v, i in ranked[-5:]])